# KPI-RAG: Explainable Root-Cause Analysis for 5G Networks
## MSc AI & Data Science — Queen's University Belfast
**Industry Partner:** Capgemini | **Supervisor:** Prof. Yuanzhu Chen

**Team:**
- Ahmed Al-Shobaki (P4) — LLM & Dashboard
- Farida El Ghabary (P1) — Data & Features
- Raneem Alnaghy (P2) — ML & SHAP
- Rodina Khallaf (P3) — RAG & Alignment Table

**Pipeline:** TelecomTS KPI signals → Random Forest detection →
SHAP attribution → RAG retrieval → LLM explanation grounded in 3GPP standards


In [ ]:
import sys, os, json, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

from src.config_loader import load_config
from src.schema import ClassifierOutput, AnomalyType
from src.utils import validate_3gpp_ref, setup_logging
from src.kg_indexer import get_collection
from src.rag_query import build_query, query_from_classifier_output
from src.llm_explainer import load_alignment_table, explain
from src.evaluator import score_explanation, compute_track_b, compute_track_c

import logging
logging.basicConfig(level=logging.WARNING)

cfg = load_config("configs/config.yaml")
print(f"Backend: {cfg['llm']['backend']}")
print(f"Embedding: {cfg['rag']['embedding_model']}")
print(f"Threshold: {cfg['rag']['cosine_threshold']}")

## Section 1 — Dataset Overview
TelecomTS: 32,000 samples, 18 KPI channels, 128 timesteps @ 100ms
- 30,765 normal samples
- 1,235 anomalous samples across 11 fault types
- Source: https://huggingface.co/datasets/AliMaatouk/TelecomTS


In [ ]:
from src.data_loader import load_jsonl_files, filter_anomalous, extract_tickets
from collections import Counter
from pathlib import Path
import pandas as pd

print("Loading TelecomTS...")
records = load_jsonl_files(cfg["data"]["raw_path"])
anomalous = filter_anomalous(records)
tickets = extract_tickets(anomalous)

print(f"Total records: {len(records):,}")
print(f"Anomalous: {len(anomalous):,}")
print(f"Normal: {len(records) - len(anomalous):,}")

fault_counts = Counter(t["anomaly_type"] for t in tickets)
df = pd.DataFrame(
    fault_counts.items(),
    columns=["Fault Type", "Count"]
).sort_values("Count", ascending=False).reset_index(drop=True)
df.index += 1
print("\nFault Type Distribution:")
print(df.to_string())

## Section 2 — ChromaDB Index
1,235 troubleshooting tickets embedded with all-MiniLM-L6-v2
and indexed in ChromaDB for semantic retrieval.


In [ ]:
collection = get_collection(cfg)
print(f"Collection: {cfg['rag']['collection_name']}")
print(f"Documents indexed: {collection.count():,}")
print(f"Embedding model: {cfg['rag']['embedding_model']}")
print(f"Cosine threshold: {cfg['rag']['cosine_threshold']}")

sample = collection.get(limit=2, include=["metadatas", "documents"])
print("\nSample indexed tickets:")
for i, (meta, doc) in enumerate(zip(
    sample["metadatas"], sample["documents"]
)):
    print(f"\n[{i+1}] Fault: {meta['anomaly_type']}")
    print(f"    Ticket ID: {meta['ticket_id']}")
    print(f"    Content preview: {doc[:120]}...")

## Section 3 — Fault-to-Standard Alignment Table
Maps 10 synthetic TelecomTS fault types to specific 3GPP Technical
Specification clauses and O-RAN components.
This is the project's primary novel contribution.


In [ ]:
alignment = load_alignment_table("configs/alignment_table.json")

rows = []
for fault, entry in alignment.items():
    rows.append({
        "Fault Type": fault,
        "3GPP TS": entry["3gpp_ts"],
        "Clause": entry["clause"],
        "O-RAN Component": entry["oran_component"],
        "Status": entry["validated_by"]
    })

df_align = pd.DataFrame(rows)
print(f"Alignment table: {len(df_align)} entries (Jamming excluded)")
print(df_align.to_string(index=False))

## Section 4 — End-to-End Pipeline Demo

### Input: ClassifierOutput from Layer 2
Simulates the JSON that Layer 2 (Random Forest + SHAP) produces.
In production this comes from Raneem's classifier.


In [ ]:
TEST_CASES = [
    {
        "name": "Antenna Failure",
        "payload": {
            "anomaly_type": "Antenna Failure",
            "confidence": 0.87,
            "shap_top3": [
                {"channel": "RSRP",    "shap_value": -0.42,
                 "direction": "below_normal"},
                {"channel": "DL_BLER", "shap_value":  0.28,
                 "direction": "above_normal"},
                {"channel": "DL_MCS",  "shap_value": -0.19,
                 "direction": "below_normal"}
            ],
            "signal_statistics": {
                "RSRP":    {"mean": -105, "std": 3.2,
                            "min": -112, "max": -98},
                "DL_BLER": {"mean": 0.35, "std": 0.08,
                            "min": 0.21, "max": 0.51}
            }
        }
    },
    {
        "name": "Buffer Overflow (Gradual Buildup)",
        "payload": {
            "anomaly_type": "Buffer Overflow (Gradual Buildup)",
            "confidence": 0.79,
            "shap_top3": [
                {"channel": "UL_BLER",    "shap_value":  0.51,
                 "direction": "above_normal"},
                {"channel": "DL_PRB_UTIL","shap_value":  0.38,
                 "direction": "above_normal"},
                {"channel": "DL_BLER",    "shap_value":  0.22,
                 "direction": "above_normal"}
            ],
            "signal_statistics": {
                "UL_BLER":     {"mean": 0.42, "std": 0.11,
                                "min": 0.28, "max": 0.61},
                "DL_PRB_UTIL": {"mean": 0.88, "std": 0.05,
                                "min": 0.79, "max": 0.95}
            }
        }
    },
    {
        "name": "Faulty Handover Algorithm (Too Frequent)",
        "payload": {
            "anomaly_type": "Faulty Handover Algorithm (Too Frequent)",
            "confidence": 0.83,
            "shap_top3": [
                {"channel": "RSRP",    "shap_value": -0.33,
                 "direction": "below_normal"},
                {"channel": "DL_MCS",  "shap_value": -0.27,
                 "direction": "below_normal"},
                {"channel": "UL_BLER", "shap_value":  0.21,
                 "direction": "above_normal"}
            ],
            "signal_statistics": {
                "RSRP":    {"mean": -98, "std": 4.1,
                            "min": -108, "max": -89},
                "UL_BLER": {"mean": 0.31, "std": 0.09,
                            "min": 0.19, "max": 0.44}
            }
        }
    }
]

print(f"Test cases defined: {len(TEST_CASES)}")
for tc in TEST_CASES:
    print(f"  - {tc['name']}")

In [ ]:
results = []

for tc in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"Processing: {tc['name']}")
    print(f"{'='*60}")

    payload = ClassifierOutput(**tc["payload"])

    query = build_query(payload)
    print(f"\nRAG Query:\n{query}")

    tickets, low_conf = query_from_classifier_output(
        payload, collection, cfg
    )
    print(f"\nRetrieved {len(tickets)} tickets | "
          f"low_confidence={low_conf}")
    for t in tickets[:2]:
        print(f"  [{t.anomaly_type}] "
              f"similarity={t.similarity_score:.3f}")

    explanation = explain(payload, tickets, cfg, alignment)

    print(f"\nRoot Cause: {explanation.root_cause}")
    print(f"3GPP Reference: {explanation.gpp_reference}")
    print(f"Reference Valid: {explanation.reference_valid}")
    print(f"O-RAN: {explanation.oran_component}")
    print(f"Action: {explanation.recommended_action}")
    print(f"Template: {explanation.template_generated}")

    results.append({
        "fault_type": tc["name"],
        "payload": payload,
        "tickets": tickets,
        "low_conf": low_conf,
        "explanation": explanation
    })

## Section 5 — Results Summary


In [ ]:
summary_rows = []
for r in results:
    exp = r["explanation"]
    summary_rows.append({
        "Fault Type": r["fault_type"],
        "Tickets Retrieved": len(r["tickets"]),
        "Top Similarity": (
            f"{max(t.similarity_score for t in r['tickets']):.3f}"
            if r["tickets"] else "N/A"
        ),
        "3GPP Ref": exp.gpp_reference,
        "Valid": "✅" if exp.reference_valid else "❌",
        "Template": "⚠️" if exp.template_generated else "✅ LLM"
    })

df_results = pd.DataFrame(summary_rows)
print("Pipeline Results Summary:")
print(df_results.to_string(index=False))

valid_count = sum(
    1 for r in results if r["explanation"].reference_valid
)
print(f"\nCitation validity rate: "
      f"{valid_count}/{len(results)} = "
      f"{valid_count/len(results):.1%}")

## Section 6 — 3GPP Citation Validation

Two-check validation system:
1. **Format regex**: series 21–38, three-digit sub-number
2. **Alignment table cross-check**: reference must be in known-valid list


In [ ]:
test_refs = [
    ("TS 38.104", "Known valid — Antenna Failure"),
    ("TS 38.331", "Known valid — Handover"),
    ("TS 38.214", "Known valid — Resource Allocation"),
    ("TS 39.104", "Invalid series (>38)"),
    ("TS 38.1",   "Invalid sub-number (<3 digits)"),
    ("TS 38.999", "Valid format, not in alignment table"),
    ("38.104",    "Missing TS prefix"),
]

valid_ts_set = {e["3gpp_ts"] for e in alignment.values()}

print(f"{'Reference':<20} {'Format':<10} {'In Table':<12} {'Result'}")
print("-" * 55)
for ref, desc in test_refs:
    fmt_ok = validate_3gpp_ref(ref)
    in_table = ref in valid_ts_set
    result = "✅ VALID" if (fmt_ok and in_table) else "❌ INVALID"
    print(f"{ref:<20} {str(fmt_ok):<10} {str(in_table):<12} {result}")
    print(f"  → {desc}")

## Summary

| Component | Status | Notes |
|---|---|---|
| TelecomTS Dataset | ✅ | 32,000 samples, 33 JSONL files |
| ChromaDB Index | ✅ | 1,235 tickets, cosine metric |
| Embedding Model | ✅ | all-MiniLM-L6-v2 (locked) |
| Alignment Table | ✅ DRAFT | 10 rows, human validation pending |
| RAG Retrieval | ✅ | Semantic search, threshold=0.35 |
| LLM Explanation | ✅ | Groq llama-3.1-8b-instant |
| Citation Validation | ✅ | Format regex + alignment cross-check |
| Dashboard | ✅ | Streamlit, 5 panels |

**Next steps:**
- Receive train_idx.npy from P1 (Farida) → rebuild index on train split only
- Calibrate cosine threshold at Week 9
- Deploy to HuggingFace Spaces
- Run G-Eval on 30 stratified samples (Track B)
- Run 3-condition ablation (Track C)
